# Dumb Baseline — stat features + classical ML

**Purpose**: find the true ceiling of cross_subject state classification using the simplest possible model. If this beats or matches neural fusions, the fusion complexity isn't pulling its weight.

**Setup**:
- 30s windows, stride 15s over each session
- Feature per window: concat of `[mean(video), std(video), mean(km), std(km), mean(telem), std(telem)]` → 1804-dim
- Label per window: majority vote of state labels in that window
- Split: `session_tvt.json` (cross_subject)
- Classifiers: LogReg, RandomForest, XGBoost (if available)
- Metric: macro_f1_state on test split, 3 seeds

**Note**: dumb baseline predicts 1 label per 30s window; neural models predict 1 label per 0.2s frame. Their macro_f1 values are not identical but close enough for ceiling estimation.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────
import os, sys, json, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch

# Colab: mount drive
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/AmuCS_experiment')
else:
    DRIVE_ROOT = Path(r'g:\我的云端硬盘\AmuCS_experiment')

FEATURES_ROOT = DRIVE_ROOT / 'features' / 'aligned'
LABELS_PATH   = DRIVE_ROOT / 'labels' / 'arousal_state_trend_seq.json'
SPLIT_PATH    = DRIVE_ROOT / 'splits' / 'session_tvt.json'

print('features root:', FEATURES_ROOT, 'exists =', FEATURES_ROOT.exists())
print('labels:', LABELS_PATH, 'exists =', LABELS_PATH.exists())
print('split:', SPLIT_PATH, 'exists =', SPLIT_PATH.exists())

In [ ]:
# ── 1. Config ─────────────────────────────────────────────
SEQ_FPS       = 5         # 5Hz aligned features
WINDOW_SEC    = 30.0
STRIDE_SEC    = 15.0
WIN_FRAMES    = int(WINDOW_SEC * SEQ_FPS)   # 150
STRIDE_FRAMES = int(STRIDE_SEC * SEQ_FPS)   # 75

MODALITIES = {
    'video': 'video_clip',  # 768-dim
    'km':    'km',          # 25-dim
    'telem': 'telem',       # 109-dim
}

SEEDS = [0, 1, 2]
TASK  = 'state'

print(f'window = {WIN_FRAMES} frames ({WINDOW_SEC}s), stride = {STRIDE_FRAMES} frames ({STRIDE_SEC}s)')

In [ ]:
# ── 2. Load split + labels ────────────────────────────────
with open(SPLIT_PATH, encoding='utf-8-sig') as f:
    split = json.load(f)
with open(LABELS_PATH, encoding='utf-8-sig') as f:
    labels_all = json.load(f)

print('split sizes:', {k: len(v) for k, v in split.items()})
print('label sessions:', len(labels_all))

# Check existing stems across modalities
def available_stems(mod_dir):
    return {p.stem for p in (FEATURES_ROOT / mod_dir).glob('*.pt')}

common = set(labels_all.keys())
for m, d in MODALITIES.items():
    common &= available_stems(d)
print(f'stems with all modalities + labels: {len(common)}')

In [ ]:
# ── 3. Feature extraction (mean+std over window, per modality) ─
def load_stem_features(stem):
    """Return dict: {mod: [T, D]} + min length T_min across modalities + state labels [T_lbl]."""
    feats = {}
    for m, d in MODALITIES.items():
        obj = torch.load(FEATURES_ROOT / d / f'{stem}.pt', map_location='cpu', weights_only=False)
        feats[m] = obj['features'].numpy().astype(np.float32)  # [T, D]
    states = np.array(labels_all[stem][TASK]['values'], dtype=np.int64)
    state_mask = np.array(labels_all[stem][TASK].get('mask', [True]*len(states)), dtype=bool)
    T_min = min([f.shape[0] for f in feats.values()] + [len(states)])
    feats = {m: f[:T_min] for m, f in feats.items()}
    states = states[:T_min]
    state_mask = state_mask[:T_min]
    return feats, states, state_mask, T_min

def window_iter(T, win, stride):
    if T < win:
        return
    for s in range(0, T - win + 1, stride):
        yield s, s + win

def build_split_matrix(stems):
    """Return X [N, D_total], y [N], stems_per_window [N], counts."""
    X_rows, y_rows, stem_rows = [], [], []
    skipped = {'no_features': 0, 'no_valid_label': 0, 'short': 0}
    for stem in sorted(stems):
        try:
            feats, states, smask, T = load_stem_features(stem)
        except FileNotFoundError:
            skipped['no_features'] += 1
            continue
        if T < WIN_FRAMES:
            skipped['short'] += 1
            continue
        for s, e in window_iter(T, WIN_FRAMES, STRIDE_FRAMES):
            w_mask = smask[s:e]
            if not w_mask.any():
                continue
            # majority vote over valid frames
            valid_states = states[s:e][w_mask]
            y = Counter(valid_states.tolist()).most_common(1)[0][0]
            # feature: mean + std per modality, concat
            parts = []
            for m in MODALITIES:
                w = feats[m][s:e]
                parts.append(w.mean(axis=0))
                parts.append(w.std(axis=0))
            X_rows.append(np.concatenate(parts))
            y_rows.append(y)
            stem_rows.append(stem)
    if not X_rows:
        return np.empty((0, 0)), np.array([]), [], skipped
    return np.stack(X_rows), np.array(y_rows), stem_rows, skipped

t0 = time.time()
splits_data = {}
for sp in ['train', 'val', 'test']:
    stems = [s for s in split[sp] if s in common]
    X, y, stems_w, skipped = build_split_matrix(stems)
    splits_data[sp] = (X, y, stems_w)
    print(f'{sp}: {len(stems)} sessions → {len(y)} windows, X shape={X.shape}, skipped={skipped}')
print(f'total load time: {time.time()-t0:.1f}s')

# Class distribution
print('\nclass distribution (test):', Counter(splits_data['test'][1].tolist()))
print('class distribution (train):', Counter(splits_data['train'][1].tolist()))

In [ ]:
# ── 4. Train classifiers × 3 seeds ───────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed; skipping XGBoost')

X_tr, y_tr, _ = splits_data['train']
X_va, y_va, _ = splits_data['val']
X_te, y_te, _ = splits_data['test']

# Sanity checks
assert X_tr.size > 0 and X_va.size > 0 and X_te.size > 0, 'empty split(s)'
tr_classes = set(y_tr.tolist())
all_classes = tr_classes | set(y_va.tolist()) | set(y_te.tolist())
missing_in_train = all_classes - tr_classes
if missing_in_train:
    print(f'WARNING: classes {missing_in_train} missing from train split; classifiers will never predict them')
print(f'classes seen (train/val/test): {sorted(tr_classes)} / {sorted(set(y_va.tolist()))} / {sorted(set(y_te.tolist()))}')

# Scale once for LogReg (seed-independent)
scaler = StandardScaler().fit(X_tr)
Xs_tr = scaler.transform(X_tr)
Xs_va = scaler.transform(X_va)
Xs_te = scaler.transform(X_te)

def run_logreg(seed):
    best = {'f1': -1, 'C': None, 'model': None}
    for C in [0.01, 0.1, 1.0, 10.0]:
        clf = LogisticRegression(
            C=C, max_iter=2000, class_weight='balanced',
            random_state=seed, n_jobs=-1,
        )
        clf.fit(Xs_tr, y_tr)
        f1_v = f1_score(y_va, clf.predict(Xs_va), average='macro')
        if f1_v > best['f1']:
            best = {'f1': f1_v, 'C': C, 'model': clf}
    y_pred = best['model'].predict(Xs_te)
    return {
        'model': 'LogReg', 'seed': seed, 'best_hp': f"C={best['C']}", 'val_f1': best['f1'],
        'test_macro_f1': f1_score(y_te, y_pred, average='macro'),
        'test_balanced_acc': balanced_accuracy_score(y_te, y_pred),
    }

def run_rf(seed):
    best = {'f1': -1, 'cfg': None, 'model': None}
    for n_est in [200, 500]:
        for max_depth in [None, 10, 20]:
            clf = RandomForestClassifier(
                n_estimators=n_est, max_depth=max_depth,
                class_weight='balanced', n_jobs=-1, random_state=seed,
            )
            clf.fit(X_tr, y_tr)
            f1_v = f1_score(y_va, clf.predict(X_va), average='macro')
            if f1_v > best['f1']:
                best = {'f1': f1_v, 'cfg': (n_est, max_depth), 'model': clf}
    y_pred = best['model'].predict(X_te)
    return {
        'model': 'RandomForest', 'seed': seed, 'best_hp': f"n={best['cfg'][0]},depth={best['cfg'][1]}",
        'val_f1': best['f1'],
        'test_macro_f1': f1_score(y_te, y_pred, average='macro'),
        'test_balanced_acc': balanced_accuracy_score(y_te, y_pred),
    }

def run_xgb(seed):
    if not HAS_XGB:
        return None
    # XGBoost has no class_weight='balanced' — emulate via sample weights
    sample_w = compute_sample_weight(class_weight='balanced', y=y_tr)
    best = {'f1': -1, 'cfg': None, 'model': None}
    n_classes = len(np.unique(y_tr))
    for n_est in [200, 500]:
        for max_depth in [4, 6]:
            for lr in [0.05, 0.1]:
                clf = XGBClassifier(
                    n_estimators=n_est, max_depth=max_depth, learning_rate=lr,
                    objective='multi:softprob', num_class=n_classes,
                    eval_metric='mlogloss', random_state=seed, n_jobs=-1,
                    verbosity=0,
                )
                clf.fit(X_tr, y_tr, sample_weight=sample_w)
                f1_v = f1_score(y_va, clf.predict(X_va), average='macro')
                if f1_v > best['f1']:
                    best = {'f1': f1_v, 'cfg': (n_est, max_depth, lr), 'model': clf}
    y_pred = best['model'].predict(X_te)
    return {
        'model': 'XGBoost', 'seed': seed,
        'best_hp': f"n={best['cfg'][0]},depth={best['cfg'][1]},lr={best['cfg'][2]}",
        'val_f1': best['f1'],
        'test_macro_f1': f1_score(y_te, y_pred, average='macro'),
        'test_balanced_acc': balanced_accuracy_score(y_te, y_pred),
    }

results = []
for seed in SEEDS:
    np.random.seed(seed)
    for fn in [run_logreg, run_rf, run_xgb]:
        t0 = time.time()
        r = fn(seed)
        if r is None:
            continue
        r['time_s'] = round(time.time()-t0, 1)
        print(f"  {r['model']:12s} seed={seed} val_f1={r['val_f1']:.4f} test_macro_f1={r['test_macro_f1']:.4f} bal_acc={r['test_balanced_acc']:.4f} hp=[{r['best_hp']}] [{r['time_s']}s]")
        results.append(r)

In [ ]:
# ── 5. Summary ───────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(results)
summary = df.groupby('model').agg(
    test_macro_f1_mean=('test_macro_f1', 'mean'),
    test_macro_f1_std=('test_macro_f1', 'std'),
    test_balanced_acc_mean=('test_balanced_acc', 'mean'),
    val_f1_mean=('val_f1', 'mean'),
).round(4)
print('=== Dumb Baseline Results (cross_subject, test set, 3 seeds) ===')
print(summary)

print('\n=== Compare to neural baselines (Path A cross_subject, test_macro_f1_state) ===')
print('  baseline gated_triple: 0.4388   (best neural)')
print('  baseline cma_triple:   0.4252')
print('  Path C cma_dual_vk_ev: 0.4299   (best Path C)')
print('\nIf dumb baseline >= 0.38, fusion complexity is not paying off.')

# Save results
OUT = DRIVE_ROOT / 'runs' / 'dumb_baseline' / 'results.csv'
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print(f'\nsaved: {OUT}')